# Chapitre 8 - Cours 3 : bagging, moyennage de modèles, stacking et bumping

Les deux premiers notebooks utilisaient le bootstrap pour **mesurer** l'incertitude. Ici
(sections 8.7 à 8.9 du livre), on s'en sert pour **améliorer** les prédictions :

1. le **bagging** (*bootstrap aggregating*) : moyenner un modèle réajusté sur des
   échantillons bootstrap (figures 8.9 à 8.12) ;
2. le **moyennage bayésien de modèles** et le **stacking** : combiner des modèles
   différents avec des poids bien choisis ;
3. le **bumping** : utiliser le bootstrap pour explorer l'espace des modèles et trouver
   un meilleur modèle unique (figure 8.13).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.plotting import decision_grid
from esl import datasets
from esl import ch08_inference as inf
set_style()
rng = np.random.RandomState(0)

## 1. Le bagging (section 8.7)

On dispose de données $Z = \{(x_1, y_1), \dots, (x_N, y_N)\}$ et d'une méthode qui
fournit une prédiction $\hat f(x)$. Pour chaque échantillon bootstrap $Z^{*b}$
($b = 1, \dots, B$), on réajuste la méthode, ce qui donne $\hat f^{*b}(x)$, puis on
moyenne :

$$\hat f_{\text{bag}}(x) = \frac1B\sum_{b=1}^B \hat f^{*b}(x). \qquad \text{(éq. (8.51))}$$

Si $\hat P$ désigne la loi empirique (masse $1/N$ sur chaque couple $(x_i, y_i)$), le
« vrai » estimateur bagging est $E_{\hat P}\hat f^*(x)$ ; (8.51) en est une
approximation de Monte-Carlo, qui converge quand $B \to \infty$.

> **Intuition.** Moyenner réduit la **variance** sans changer (beaucoup) le biais. Le
> bagging n'a d'intérêt que si $\hat f$ dépend de façon **non linéaire** ou
> **instable** des données : pour un lisseur linéaire, il ne change presque rien
> (exercice 8.4).

### Bagging d'un classifieur : votes ou probabilités ?

Pour un problème à $K$ classes, on associe au classifieur $\hat G(x)$ le vecteur
indicateur $\hat f(x)$ (un 1 à la classe prédite, des 0 ailleurs). Alors
$\hat f_{\text{bag}}(x) = (p_1(x), \dots, p_K(x))$ où $p_k(x)$ est la **proportion
d'arbres qui votent** pour la classe $k$, et le classifieur bagging est
$\hat G_{\text{bag}}(x) = \arg\max_k \hat f_{\text{bag}}(x)$ (**vote majoritaire**, ou
consensus).

Les proportions de votes **ne sont pas** des estimations des probabilités de classe : si
$\Pr(Y = 1\mid x) = 0{,}75$ et que tous les classifieurs prédisent 1, on obtient
$p_1(x) = 1$. Si le classifieur fournit lui-même des probabilités (pour un arbre : les
proportions de classes dans la feuille), on peut plutôt **moyenner ces probabilités**.